In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64

# Configure OS routines
import os
import sys

# Configure the data routines
import pandas as pd

JupyterDash.infer_jupyter_proxy_config()

# Import the CRUD module from Project One (lives in the code_files folder).
# The path has to be set before the import, so this import comes last.
os.chdir('/home/codio/workspace/code_files')
sys.path.insert(0, '/home/codio/workspace/code_files')
from CRUD_Python_Module import AnimalShelter  # noqa: E402


###########################
# Data Manipulation / Model
###########################
# Credentials for the aacuser account. Host, port, database, and collection
# use the defaults set in the AnimalShelter constructor (localhost, 27017, aac, animals)
username = "aacuser"
password = "Pass1805"
shelter = AnimalShelter(username, password)

# Rescue type filters built from the Rescue Type and Preferred Dog Breeds
# table in the Dashboard Specifications Document. Each entry holds the
# preferred breeds, preferred sex, and training age range (in weeks).
RESCUE_FILTERS = {
    'Water': {
        'breeds': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever',
                   'Newfoundland'],
        'sex': 'Intact Female',
        'min_weeks': 26,
        'max_weeks': 156
    },
    'Mountain': {
        'breeds': ['German Shepherd', 'Alaskan Malamute',
                   'Old English Sheepdog', 'Siberian Husky', 'Rottweiler'],
        'sex': 'Intact Male',
        'min_weeks': 26,
        'max_weeks': 156
    },
    'Disaster': {
        'breeds': ['Doberman Pinscher', 'German Shepherd',
                   'Golden Retriever', 'Bloodhound', 'Rottweiler'],
        'sex': 'Intact Male',
        'min_weeks': 20,
        'max_weeks': 300
    }
}


def build_query(filter_type):
    """
    Build the MongoDB query for the selected rescue type.

    filter_type -- 'Water', 'Mountain', 'Disaster', or 'Reset'
    Returns a query dict. Reset (or anything unknown) returns {} so
    all documents come back unfiltered.
    """
    rescue = RESCUE_FILTERS.get(filter_type)
    if rescue is None:
        return {}

    # The age range lives in ONE dict. Repeating the key twice would make
    # Python keep only the last value and silently drop the minimum age.
    return {
        'animal_type': 'Dog',
        'breed': {'$in': rescue['breeds']},
        'sex_upon_outcome': rescue['sex'],
        'age_upon_outcome_in_weeks': {
            '$gte': rescue['min_weeks'],
            '$lte': rescue['max_weeks']
        }
    }


def get_dataframe(query):
    """
    Run a query through the CRUD module and return the results as a dataframe.

    query -- MongoDB query dict passed to AnimalShelter.read()
    Returns a dataframe without the '_id' column (empty if nothing matched).
    """
    try:
        results = pd.DataFrame.from_records(shelter.read(query))
    except Exception as error:
        print(f"Could not load data from MongoDB: {error}")
        return pd.DataFrame()

    # MongoDB returns an '_id' column of type ObjectId, which the DataTable
    # cannot display, so it is dropped. errors='ignore' covers empty results.
    results = results.drop(columns=['_id'], errors='ignore')

    # The collection holds repeated copies of the same records from an earlier
    # import. With '_id' gone the copies are identical, so remove them here.
    return results.drop_duplicates().reset_index(drop=True)


# Send the read method an empty query to retrieve all documents (unfiltered view)
df = get_dataframe({})

# Debug testing
# print(len(df.to_dict(orient='records')))
# print(df.columns.tolist())


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare logo, encoded so it can be embedded directly in the page.
# If the file can't be found, the dashboard still loads with a text fallback.
image_filename = 'Grazioso Salvare Logo.png'
try:
    with open(image_filename, 'rb') as image_file:
        encoded_image = base64.b64encode(image_file.read()).decode()
    logo = html.Img(src='data:image/png;base64,{}'.format(encoded_image),
                    style={'height': '120px'})
except FileNotFoundError:
    print(f"Logo file '{image_filename}' not found")
    logo = html.B('Grazioso Salvare')

app.layout = html.Div([
    # Branding: logo wrapped in an anchor tag to the client's home page
    html.Center(html.A(logo, href='https://www.snhu.edu', target='_blank')),
    html.Center(html.B(html.H1('SNHU CS-340 Dashboard'))),
    # Unique identifier displayed for proof of execution
    html.Center(html.H4('Grazioso Salvare Dashboard | Developed by Carrie Cummings (SNHU CS-340)',
                        id='unique-id',
                        style={'color': '#1F4E79', 'fontStyle': 'italic'})),
    html.Hr(),

    # Interactive filtering options. Radio items allow only one rescue type
    # at a time, which keeps the filter simple and reduces user errors.
    html.Div([
        html.B('Filter by Rescue Type: '),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster Rescue or Individual Tracking', 'value': 'Disaster'},
                {'label': 'Reset', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'marginRight': '20px'}
        )
    ], style={'textAlign': 'center'}),
    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        # Interactive features to make the table easier for the client to use
        page_action='native',           # Split records into pages
        page_current=0,                 # Start on the first page
        page_size=10,                   # Show 10 rows per page
        sort_action='native',           # Click column headers to sort
        filter_action='native',         # Type in the filter row to search a column
        row_selectable='single',        # Single row selection (drives the map)
        selected_rows=[0],              # Select the first row by default
        column_selectable='single',     # Allow one column to be highlighted
        style_table={'overflowX': 'auto'},  # Horizontal scroll for wide data
        style_cell={'textAlign': 'left', 'padding': '5px'},
        style_header={'fontWeight': 'bold', 'backgroundColor': '#E8E8E8'}
    ),
    html.Br(),
    html.Hr(),

    # This sets up the dashboard so that the pie chart and the geolocation
    # chart are side-by-side
    html.Div(className='row',
             style={'display': 'flex'},
             children=[
                 html.Div(
                     id='graph-id',
                     className='col s12 m6',
                 ),
                 html.Div(
                     id='map-id',
                     className='col s12 m6',
                 )
             ])
])


#############################################
# Interaction Between Components / Controller
#############################################
# This callback runs the rescue type query through the CRUD module whenever
# a radio item is selected, then refreshes the table. It also resets the
# table to page one and selects the first row so the map has a valid entry.
@app.callback(
    [Output('datatable-id', 'data'),
     Output('datatable-id', 'page_current'),
     Output('datatable-id', 'selected_rows')],
    [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    dff = get_dataframe(build_query(filter_type))
    return dff.to_dict('records'), 0, [0]


# This callback builds a pie chart of the breeds currently showing in the
# data table. Because it reads derived_virtual_data, it follows the radio
# filters and any sorting or column filtering done inside the table.
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    # Nothing to chart until the table has data
    if not viewData:
        return [html.P('No matching animals to chart.')]

    dff = pd.DataFrame.from_dict(viewData)

    # The unfiltered data has hundreds of breeds, which makes an unreadable
    # pie. Keep the top 10 breeds and group the rest into 'Other'.
    breed_counts = dff['breed'].value_counts()
    if len(breed_counts) > 10:
        top_breeds = breed_counts.head(10)
        top_breeds['Other'] = breed_counts.iloc[10:].sum()
        breed_counts = top_breeds

    chart_data = breed_counts.reset_index()
    chart_data.columns = ['breed', 'count']

    return [
        dcc.Graph(
            figure=px.pie(chart_data, names='breed', values='count',
                          title='Breeds in Current View')
        )
    ]


# This callback highlights a column on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # No column has been selected yet when the page first loads
    if not selected_columns:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback updates the geolocation chart for the selected data entry.
# derived_virtual_data is the data currently in the table (after any sort/filter)
# as a list of dictionaries. derived_virtual_selected_rows is a list holding the
# selected row index. Only single row selection is allowed, so there is one value.
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    # Nothing to map until the table has data
    if not viewData:
        return [html.P('No matching animals to map.')]

    dff = pd.DataFrame.from_dict(viewData)

    # Default to the first row if nothing is selected or the selection
    # is no longer in the current (filtered) view
    if not index or index[0] >= len(dff):
        row = 0
    else:
        row = index[0]

    lat = dff.iloc[row]['location_lat']
    lon = dff.iloc[row]['location_long']

    # Center the map on the selected animal (Austin TX area)
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[lat, lon], zoom=10, children=[
                   dl.TileLayer(id="base-layer-id"),
                   # Marker placed at the animal's latitude and longitude
                   dl.Marker(position=[lat, lon],
                             children=[
                                 # Tooltip shows the breed on hover
                                 dl.Tooltip(dff.iloc[row]['breed']),
                                 # Popup shows the animal's name on click
                                 dl.Popup([
                                     html.H1("Animal Name"),
                                     html.P(dff.iloc[row]['name'])
                                 ])
                             ])
               ])
    ]


# Run app and display result in jupyterlab mode. If port 8050 is already in use
# from a prior run, set an alternate port, e.g. app.run_server(port=8051)
app.run_server()

Dash app running on https://protonloyal-providesonata-3000.codio.io/proxy/8050/
